# This notebook covers how to use the aligned Stack model to perform in-context drug perturbation prediction tasks.

In [1]:
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import os, sys

## Download the example checkpoint and anndata from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download

repo_id = "arcinstitute/Stack-Large-Aligned"
target_dir = "tutorial-pred-model"

if os.path.exists(target_dir) and os.listdir(target_dir):
    raise SystemExit(f"'{target_dir}' already exists and is not empty.")

snapshot_download(
    repo_id=repo_id,
    repo_type="model",
    revision="main",
    local_dir=target_dir,
    local_dir_use_symlinks=False,  
    resume_download=True,
)

print(f"Model downloaded to ./{target_dir}")

repo_id = "arcinstitute/Stack-DrugPBMC-Example"
target_dir = "tutorial-pred-data"

if os.path.exists(target_dir) and os.listdir(target_dir):
    raise SystemExit(f"'{target_dir}' already exists and is not empty.")

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    revision="main",
    local_dir=target_dir,
    local_dir_use_symlinks=False,  
    resume_download=True,
)

print(f"Data downloaded to ./{target_dir} ")

This example uses Donor 1 data from the 2023 OpenProblems.Bio competition. Mirroring the original competition setup, we hold out myeloid and B cells and predict their expression from T cells.

The context AnnData consists of T cells under each drug-perturbation condition, while the test AnnData contains control myeloid and B cells. To remove artifacts from differential expression predictions, we apply a synthetic control approach using control T cells as the reference.

In [2]:
adata = sc.read_h5ad('tutorial-pred-data/openproblems_donor1.h5ad')
all_context_adata = adata[adata.obs['broad_cell_class'].isin(['t cell'])]
ctrl_adata = adata[adata.obs['broad_cell_class'].isin(['t cell'])]
test_adata = adata[(~adata.obs['broad_cell_class'].isin(['t cell'])) & (adata.obs['sm_name']=='Dimethyl Sulfoxide')]
all_context_adata.write('tutorial-pred-data/openproblems_context.h5ad')
test_adata.write('tutorial-pred-data/openproblems_test.h5ad')

Now we can use Stack CLI command to predict the unseen expressions of perturbed myeloid and B cells without training.

In [4]:
! stack-generation --checkpoint "tutorial-pred-model/bc_large_aligned.ckpt" \
--base-adata "tutorial-pred-data/openproblems_context.h5ad" --test-adata "tutorial-pred-data/openproblems_test.h5ad" \
--genelist "tutorial-pred-model/basecount_1000per_15000max.pkl" --split-column sm_name --batch-size 16 --num-steps 5 \
--output-dir tutorial-pred-output --prompt-ratio 0.25 --context-ratio 0.25

[KeOps] Warning : CUDA was detected, but driver API could not be initialized. Switching to CPU only.
INFO:stack.model_loading:Loading model from tutorial-pred-model/bc_large_aligned.ckpt
/home/mingze/shiftab/state-ICL/src/stack/model_loading.py:47: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't ha

The output folder tutorial-pred-output now contains one AnnData file for each perturbation condition, as well as the control condition (Dimethyl Sulfoxide). We can then perform differential expression analysis on these datasets. This framework is highly generalizable. For instance, by using Parse perturbation conditions as context data and Tabula Sapiens as test data, it can generate perturbed expression profiles across all Tabula Sapiens tissues and cell types.